[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traceopt-ai/traceml/blob/main/notebooks/lightning_dataloading_bottleneck.ipynb)

# Find a data-loading bottleneck with PyTorch Lightning

A Lightning `Trainer` can report steady progress while its GPU waits for the next batch at every step. This notebook answers a practical question:

> Can a better DataLoader configuration reduce input wait and keep the GPU busy, and what does the run average hide?

You will train the same ResNet-18 Lightning job twice on the 320px Imagenette dataset. TraceML's `TraceMLCallback` diagnoses each run, and `traceml compare` shows whether the loader change improved training time. A last section reads the per-step evidence of the faster run to show a cost that no run-level average reports: the first batch of every epoch.

The result is hardware-dependent. A machine with four CPU cores can feed the GPU completely, while a two-core Colab runtime may remain input-bound after tuning.

**Before you start:** switch to a GPU runtime via *Runtime -> Change runtime type -> Hardware accelerator: GPU*. TraceML is open source: `pip install traceml-ai` ([github.com/traceopt-ai/traceml](https://github.com/traceopt-ai/traceml)).

## The comparison

| Profile | DataLoader workers | Pinned memory | Persistent workers |
|---|---:|---:|---:|
| Baseline | 0 | No | No |
| Optimized | Up to 4 | Yes | Yes |

The model, images, augmentation, batch size, seed, and optimizer-step count stay fixed. The optimized profile changes the loader settings as one practical configuration, so this experiment measures the profile as a whole rather than attributing the result to one setting. Mixed precision stays off in both runs; it would change compute time and confound the comparison.

TraceML records the DataLoader wait (Input Wait), the host-to-device copy (H2D), forward, backward, and optimizer time separately for every Lightning batch. `traceml compare` shows where the difference between two runs occurred.

## Choose a run mode

`extended` is the default GPU/Imagenette demonstration. `smoke` is a small CPU-only verification mode for contributors and CI: it uses synthetic data, downloads nothing, runs the same `Trainer` and `TraceMLCallback`, and checks that TraceML produces a complete input-bound diagnosis.

In [ ]:
import os

# Change the default to "smoke" to run the small CPU-only path manually.
RUN_MODE = os.environ.get("TRACEML_NOTEBOOK_MODE", "extended")
if RUN_MODE not in {"extended", "smoke"}:
    raise ValueError("TRACEML_NOTEBOOK_MODE must be 'extended' or 'smoke'")
print(f"TraceML notebook mode: {RUN_MODE}")

## 1. Check the GPU (extended mode only)

In [ ]:
import torch

if RUN_MODE == "extended":
    !nvidia-smi -L
    print("CUDA available:", torch.cuda.is_available())
    assert (
        torch.cuda.is_available()
    ), "No GPU. Runtime -> Change runtime type -> GPU, then rerun."
else:
    print("Smoke mode uses CPU; no GPU is required.")

## 2. Install TraceML with the Lightning extra

Colab supplies a CUDA-matched PyTorch and torchvision build, so this cell installs TraceML, its runtime dependencies, and Lightning without replacing them. Outside Colab, install the extras shown in the comment below.

In [ ]:
import os

if os.environ.get("TRACEML_NOTEBOOK_SKIP_INSTALL") == "1":
    print("Notebook smoke runner provided TraceML dependencies.")
else:
    %pip install -q "traceml-ai[lightning]"
# Outside Colab or in a fresh CPU environment: %pip install -q "traceml-ai[torch,lightning]"

## 3. Get the data (extended mode only)

The extended demonstration uses the 320px Imagenette train split (326 MB, 9,469 JPEG files in 10 classes) through `torchvision.datasets.Imagenette`. JPEG decoding is the CPU cost this experiment investigates. Smoke mode skips the download and creates its data in memory.

In [ ]:
import os

if RUN_MODE == "extended":
    import torchvision

    train = torchvision.datasets.Imagenette(
        "data", split="train", size="320px", download=True
    )
    print("CPU cores:", os.cpu_count())
    print("train images:", len(train), "classes:", len(train.classes))
else:
    print("Smoke mode uses a small in-memory synthetic dataset.")

## 4. The complete Lightning script

A standard `LightningModule` and `LightningDataModule`. The TraceML additions are the two lines marked in the script: `traceml_lightning.init()` once, before the loaders exist, and `TraceMLCallback()` in the `Trainer` callbacks. The `--profile` flag flips the DataLoader settings and nothing else; `--smoke` runs the CPU verification path.

In [ ]:
%%writefile lightning_dataloading_bottleneck.py
"""Find a data-loading bottleneck in a PyTorch Lightning run.

Trains ResNet-18 on the 320px Imagenette train split under one ``Trainer``.
The ``--profile`` flag changes the DataLoader settings and nothing else, so
two runs measure the loader change alone. TraceML reports whether each run
waited on input or on compute; ``traceml compare`` shows where the
difference occurred.

Launch through ``traceml run`` (a bare ``python`` run trains untraced):

    traceml run --mode summary --logs-dir logs --run-name lightning_baseline \\
        lightning_dataloading_bottleneck.py \\
        --args --profile baseline --max-steps 300 --batch-size 64
    traceml run --mode summary --logs-dir logs --run-name lightning_optimized \\
        lightning_dataloading_bottleneck.py \\
        --args --profile optimized --max-steps 300 --batch-size 64
    traceml compare logs/lightning_baseline/final_summary.json \\
        logs/lightning_optimized/final_summary.json

CPU-only check without the dataset (what the notebook smoke job runs):

    traceml run --mode summary --logs-dir logs --run-name lightning_smoke \\
        lightning_dataloading_bottleneck.py \\
        --args --smoke --max-steps 8 --batch-size 4

The 320px Imagenette archive (326 MB) is downloaded into ``--data-dir`` on
first use through ``torchvision.datasets.Imagenette``.
"""

from __future__ import annotations

import argparse
import os
import time

import lightning as L
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset

from traceml_ai.integrations import lightning as traceml_lightning

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
SEED = 42
NUM_CLASSES = 10
IMAGENETTE_SIZE = "320px"
SMOKE_SAMPLES = 32
SMOKE_DELAY_S = 0.05


class SlowSyntheticImages(Dataset):
    """Small CPU-only dataset with deliberate fetch latency for smoke mode."""

    classes = ("zero", "one")

    def __init__(self, samples=SMOKE_SAMPLES, delay_s=SMOKE_DELAY_S):
        self.images = torch.zeros(samples, 3, 32, 32)
        self.labels = torch.arange(samples) % len(self.classes)
        self.delay_s = delay_s

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        time.sleep(self.delay_s)
        return self.images[index], self.labels[index]


def loader_settings(
    profile,
    smoke=False,
    num_workers=None,
    persistent_workers=None,
):
    """
    Resolve the DataLoader knobs for one run.

    The profile is the one experimental change: ``baseline`` decodes every
    batch in the training process, ``optimized`` lets up to four workers
    decode ahead, pins batches and keeps the workers alive between epochs.
    The two explicit arguments override the profile for extra runs. Smoke
    mode always runs single-process on CPU.
    """
    optimized = profile == "optimized"
    if smoke:
        workers = 0
    elif num_workers is not None:
        workers = int(num_workers)
    else:
        # Match workers to CPU cores; more workers than cores thrash
        # instead of overlapping (free Colab has two).
        workers = min(4, os.cpu_count() or 2) if optimized else 0
    if persistent_workers is None:
        persistent = optimized
    else:
        persistent = bool(persistent_workers)
    return {
        "num_workers": workers,
        "pin_memory": optimized and not smoke,
        "persistent_workers": persistent and workers > 0,
    }


class ImagenetteDataModule(L.LightningDataModule):
    """Imagenette train loader whose only knobs are the DataLoader settings."""

    def __init__(
        self,
        data_dir,
        batch_size,
        num_workers,
        pin_memory,
        persistent_workers,
        smoke=False,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.dataset = None

    @property
    def num_classes(self):
        if self.hparams.smoke:
            return len(SlowSyntheticImages.classes)
        return NUM_CLASSES

    def prepare_data(self):
        # Download only; Lightning calls this once per node before setup().
        if not self.hparams.smoke:
            torchvision.datasets.Imagenette(
                self.hparams.data_dir,
                split="train",
                size=IMAGENETTE_SIZE,
                download=True,
            )

    def setup(self, stage=None):
        if self.hparams.smoke:
            self.dataset = SlowSyntheticImages()
            return
        transform = T.Compose(
            [
                T.RandomResizedCrop(224),
                T.RandomHorizontalFlip(),
                T.ToTensor(),
                T.Normalize(MEAN, STD),
            ]
        )
        self.dataset = torchvision.datasets.Imagenette(
            self.hparams.data_dir,
            split="train",
            size=IMAGENETTE_SIZE,
            transform=transform,
        )

    def train_dataloader(self):
        return DataLoader(
            self.dataset,
            batch_size=self.hparams.batch_size,
            shuffle=True,
            num_workers=self.hparams.num_workers,
            pin_memory=self.hparams.pin_memory,
            persistent_workers=self.hparams.persistent_workers,
            drop_last=True,
        )


class LitResNet18(L.LightningModule):
    """ResNet-18 from scratch; the model is not the experimental variable."""

    def __init__(self, num_classes, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.model = torchvision.models.resnet18(
            weights=None, num_classes=num_classes
        )

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        return F.cross_entropy(self(images), labels)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


def build_parser():
    parser = argparse.ArgumentParser(
        description=(
            "ResNet-18 on Imagenette under Lightning; only the DataLoader "
            "profile changes between runs."
        ),
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument(
        "--profile",
        choices=["baseline", "optimized"],
        default="baseline",
        help="DataLoader profile, the only experimental change.",
    )
    parser.add_argument(
        "--data-dir",
        default="data",
        help="Where torchvision stores the Imagenette archive.",
    )
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--max-steps", type=int, default=300)
    parser.add_argument(
        "--num-workers",
        type=int,
        default=None,
        help="Override the profile's worker count.",
    )
    parser.add_argument(
        "--persistent-workers",
        action=argparse.BooleanOptionalAction,
        default=None,
        help="Override the profile's persistent_workers setting.",
    )
    parser.add_argument(
        "--smoke",
        action="store_true",
        help="CPU-only synthetic verification path; downloads nothing.",
    )
    return parser


def main(argv=None):
    args = build_parser().parse_args(argv)
    settings = loader_settings(
        args.profile,
        smoke=args.smoke,
        num_workers=args.num_workers,
        persistent_workers=args.persistent_workers,
    )
    accelerator = "cpu" if args.smoke else "auto"
    L.seed_everything(SEED, workers=True)

    traceml_lightning.init()  # TraceML line 1: fetch and H2D timers.

    datamodule = ImagenetteDataModule(
        args.data_dir, args.batch_size, smoke=args.smoke, **settings
    )
    model = LitResNet18(datamodule.num_classes)
    print(
        f"[demo] profile={args.profile} smoke={args.smoke} "
        f"accelerator={accelerator} num_workers={settings['num_workers']} "
        f"pin_memory={settings['pin_memory']} "
        f"persistent_workers={settings['persistent_workers']} "
        f"batch={args.batch_size} max_steps={args.max_steps} amp=off",
        flush=True,
    )
    trainer = L.Trainer(
        max_steps=args.max_steps,
        accelerator=accelerator,
        devices=1,
        # AMP stays off in both profiles: it would change compute time and
        # confound a comparison whose only variable is the loader.
        precision="32-true",
        enable_progress_bar=False,
        enable_checkpointing=False,
        enable_model_summary=False,
        logger=False,
        limit_val_batches=0,
        num_sanity_val_steps=0,
        callbacks=[traceml_lightning.TraceMLCallback()],  # TraceML line 2
    )
    trainer.fit(model, datamodule=datamodule)


if __name__ == "__main__":
    main()


## 5. Run the baseline

Extended mode runs 300 optimizer steps at batch 64 with `num_workers=0`, so every batch is decoded in the training process while the GPU waits. Smoke mode runs eight CPU-only synthetic steps with deliberate input delay; it should produce a complete `INPUT-BOUND` diagnosis.

Always launch through `traceml run`: it starts the aggregator that receives the callback's measurements and writes the summary. A bare `python` run trains untraced.

In [ ]:
if RUN_MODE == "smoke":
    !traceml run --mode summary --logs-dir logs --run-name lightning_smoke lightning_dataloading_bottleneck.py --args --smoke --max-steps 8 --batch-size 4
else:
    !traceml run --mode summary --logs-dir logs --run-name lightning_baseline lightning_dataloading_bottleneck.py --args --profile baseline --data-dir data --max-steps 300 --batch-size 64

## 6. Run the optimized loader (extended mode only)

Same images, model, batch size, and steps. This profile lets up to four CPU workers decode ahead, pins batches for a faster transfer, and keeps the workers alive between epochs. If your CPU can hide the decode work, the verdict flips to `COMPUTE-BOUND`.

In [ ]:
if RUN_MODE == "extended":
    !traceml run --mode summary --logs-dir logs --run-name lightning_optimized lightning_dataloading_bottleneck.py --args --profile optimized --data-dir data --max-steps 300 --batch-size 64
else:
    print(
        "Smoke mode runs one intentional input-bound case, not a comparison."
    )

## 7. Compare the two runs

TraceML writes a portable `final_summary.json` for each run. `traceml compare` prints a compact comparison and writes JSON and text artifacts for the baseline-versus-optimized result.

On one T4 with four CPU cores, GPU Step Time went from 218.7 ms to 180.1 ms (-17.6%), GPU utilization from 80% to 100%, and the diagnosis from `INPUT-BOUND` to `COMPUTE-BOUND`. The 320px images decode faster than full-resolution ones, so most of the baseline's decode already overlapped the previous step's GPU work; the compare card's Input Wait is the time the GPU actually sat idle for input on the GPU clock, and the `DataLoader Fetch (CPU)` row is how long each `next(loader)` took on the training process.

In [ ]:
import json
from pathlib import Path

if RUN_MODE == "smoke":
    summary_dir = Path("logs/lightning_smoke")
    summary_json = summary_dir / "final_summary.json"
    summary_text = summary_dir / "final_summary.txt"
    assert summary_json.is_file(), f"Missing {summary_json}"
    assert summary_text.is_file(), f"Missing {summary_text}"
    summary = json.loads(summary_json.read_text())
    diagnosis = summary["primary_diagnosis"]["status"]
    assert (
        diagnosis == "INPUT-BOUND"
    ), f"Expected the intentional input-bound smoke diagnosis, got {diagnosis!r}"
    print(f"Smoke check passed: {diagnosis}; artifacts are in {summary_dir}")
else:
    !traceml compare logs/lightning_baseline/final_summary.json logs/lightning_optimized/final_summary.json --output=logs/lightning_baseline_vs_optimized

## 8. What the run average hides: the first batch of every epoch

`traceml compare` reports averages over the whole run. Lightning creates a new iterator over the training DataLoader at the start of every epoch, and the first batch of an epoch is never decoded ahead: the workers start producing only once the epoch loop asks for a batch. On this dataset an epoch at batch 64 is 147 steps (9,469 images, `drop_last=True`), so steps 148 and 295 of the optimized run wait for a cold pipeline, while every other step reads a batch that a worker prepared during the previous step.

That wait is real time on the training process, and it does not show in the averages: two steps out of 300 barely move a mean. On one T4 with four CPU cores the optimized run's median fetch wait was 0.4 ms, while step 148 waited 340 ms and step 295 waited 384 ms for their first batch.

TraceML keeps every step's phases in the run's telemetry database next to the summary, so the cell below reads the per-step fetch wait of the optimized run (the CPU-side time of each `next(loader)`, the same quantity as the compare card's `DataLoader Fetch (CPU)` row) and prints the slowest steps. In smoke mode it reads the eight smoke steps instead.

In [ ]:
import json
import sqlite3
from pathlib import Path

PREFIX = "_traceml_internal:"
run = "lightning_smoke" if RUN_MODE == "smoke" else "lightning_optimized"


def input_wait_per_step(run_name):
    # One row per step; the fetch is a CPU wait, so read its CPU clock.
    db = Path("logs") / run_name / "aggregator" / "telemetry"
    rows = []
    with sqlite3.connect(db) as conn:
        query = (
            "SELECT step, events_json FROM step_time_samples "
            "WHERE rank = 0 ORDER BY step"
        )
        for step, events_json in conn.execute(query):
            fetch = json.loads(events_json).get(PREFIX + "dataloader_next", {})
            wait_ms = sum(d.get("cpu_ms") or 0.0 for d in fetch.values())
            rows.append((step, wait_ms))
    return rows


def report(run_name, top=5):
    rows = input_wait_per_step(run_name)
    waits = sorted(w for _, w in rows)
    median = waits[len(waits) // 2]
    print(f"{run_name}: {len(rows)} steps, median Input Wait {median:.1f} ms")
    print("  slowest steps by Input Wait:")
    for step, wait in sorted(rows, key=lambda r: r[1], reverse=True)[:top]:
        print(f"    step {step:>4}: {wait:8.1f} ms")
    return rows


rows = report(run)

import matplotlib.pyplot as plt

steps, waits = zip(*rows)
plt.figure(figsize=(9, 3))
plt.plot(steps, waits, linewidth=1)
plt.xlabel("step")
plt.ylabel("Input Wait (ms)")
plt.title(f"{run}: DataLoader wait per step")
plt.tight_layout()
plt.show()

### Does `persistent_workers` remove it? (extended mode only)

Keeping the workers alive between epochs (`persistent_workers=True`, the optimized profile) saves the worker start-up at every epoch boundary, but the first batch still has to be decoded after the iterator resets. The run below keeps everything from the optimized profile and only turns `persistent_workers` off, then prints the slowest steps of both runs. On one T4 with four CPU cores the two runs took the same wall time; per step, the non-persistent run waited 480 ms and 396 ms at steps 148 and 295 against 340 ms and 384 ms with persistent workers. Keeping the workers alive removes their restart, not the cold first batch.

Short epochs make this matter: at 147 steps per epoch the two cold steps cost 1.1% of the optimized run's wall time; a dataset that fits in ten batches per epoch pays a cold start every tenth step.

In [ ]:
if RUN_MODE == "extended":
    !traceml run --mode summary --logs-dir logs --run-name lightning_optimized_nonpersistent lightning_dataloading_bottleneck.py --args --profile optimized --no-persistent-workers --data-dir data --max-steps 300 --batch-size 64
    for name in ("lightning_optimized", "lightning_optimized_nonpersistent"):
        report(name)
else:
    print(
        "Smoke mode runs one intentional input-bound case, not a comparison."
    )

## Use this in your own Trainer

The experiment is a pattern, not a special ResNet trick. In an existing Lightning script, call `traceml_lightning.init()` once before the loaders exist and add `traceml_lightning.TraceMLCallback()` to the `Trainer` callbacks. Launch it through `traceml run --mode summary ...`; for several GPUs pair `--nproc-per-node N` with `Trainer(devices=N)`. Fetches of validation, sanity-check, test, and predict loaders are not counted as Input Wait.

If TraceML says `INPUT-BOUND`, start by testing `num_workers`, `pin_memory`, and `persistent_workers` one change at a time. Keep the change only when the measured wall time and Input Wait improve on the hardware that will actually run your job. The per-step database read in section 8 is the same for every run, so it also shows a stall that a slow shard, a network file system, or a cache miss adds to particular steps.

Guide: [PyTorch Lightning integration](https://github.com/traceopt-ai/traceml/blob/main/docs/user_guide/integrations/lightning.md).